# Super-Resolution Preprocessing (CodeFormer)

Restores/upscales low-resolution or distant faces before they're passed to face recognition, using [CodeFormer](https://github.com/sczhou/CodeFormer) — chosen over GFPGAN and Real-ESRGAN alone for better identity preservation on degraded faces.

Runs on Colab GPU. This is a preprocessing step, not a standalone module — output feeds into the face recognition stage when a face is too low-quality to match confidently.

## 1 - Environment check & basicsr compatibility shim

`basicsr` (a CodeFormer dependency) imports a torchvision internal (`functional_tensor`) that was removed in newer torchvision versions. This shim must run **before** `basicsr` is installed/imported, or the import will fail.

In [ ]:
!nvidia-smi

# Shim for basicsr's broken torchvision import (must run before installing/importing basicsr)
import sys
import types
import torchvision.transforms.functional as F

shim = types.ModuleType("torchvision.transforms.functional_tensor")
shim.rgb_to_grayscale = F.rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = shim

!pip install -q basicsr facexlib

import basicsr
print("basicsr imported successfully:", basicsr.__version__)

## 2 - Clone CodeFormer & install

In [ ]:
import os

if not os.path.exists('CodeFormer'):
    !git clone -q https://github.com/sczhou/CodeFormer.git

%cd CodeFormer
!pip install -q -r requirements.txt
!python basicsr/setup.py develop
%cd ..

print("CodeFormer setup done.")

## 3 - Download model weights

Downloads the official CodeFormer, face detection, face parsing, and Real-ESRGAN (background upsampling) weights.

**Known issue:** the official `RealESRGAN_x2plus.pth` release link is occasionally unreliable/corrupted when fetched from Colab. If the primary download comes back too small (a corrupted/partial file), this falls back to a HuggingFace mirror automatically.

In [ ]:
import os

os.makedirs('CodeFormer/weights/CodeFormer', exist_ok=True)
os.makedirs('CodeFormer/weights/facelib', exist_ok=True)
os.makedirs('CodeFormer/weights/realesrgan', exist_ok=True)

!wget -q -O CodeFormer/weights/CodeFormer/codeformer.pth https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth
!wget -q -O CodeFormer/weights/facelib/detection_Resnet50_Final.pth https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/detection_Resnet50_Final.pth
!wget -q -O CodeFormer/weights/facelib/parsing_parsenet.pth https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/parsing_parsenet.pth
!wget -q -O CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x2plus.pth

print("Weights downloaded.")

# Verify the Real-ESRGAN weight downloaded correctly; fall back to a mirror if not.
weight_path = 'CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth'
size = os.path.getsize(weight_path)
print("RealESRGAN_x2plus.pth size:", size, "bytes")

if size < 1_000_000:  # corrupted/partial download
    print("Primary download looks corrupted, trying HuggingFace mirror...")
    !rm -f {weight_path}
    !wget -q -O {weight_path} "https://huggingface.co/dtarnow/UPscaler/resolve/main/RealESRGAN_x2plus.pth"
    size = os.path.getsize(weight_path)
    print("Retry size:", size, "bytes")

## 4 - `torch.load` compatibility patch

Newer PyTorch versions default `torch.load` to `weights_only=True`, which breaks basicsr's Real-ESRGAN loader (it needs to unpickle a full checkpoint, not just tensors). This patches the loader in-place to explicitly request `weights_only=False`.

Safe to run every time — it checks whether the pattern is already patched before touching the file.

In [ ]:
import glob

files_to_patch = glob.glob('/content/CodeFormer/**/realesrgan_utils.py', recursive=True)
print("Found files:", files_to_patch)

old = "torch.load(model_path, map_location=torch.device('cpu'))"
new = "torch.load(model_path, map_location=torch.device('cpu'), weights_only=False)"

for fpath in files_to_patch:
    with open(fpath, 'r') as f:
        content = f.read()

    if old in content:
        content = content.replace(old, new)
        with open(fpath, 'w') as f:
            f.write(content)
        print(f"Patched: {fpath}")
    else:
        print(f"Already patched or pattern not found: {fpath}")

## 5 - Load input image

Swap `input_path` for your own test image (mounted from Drive here, or upload directly via `google.colab.files.upload()`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

input_path = "/content/drive/MyDrive/facial-recognition-ai/test-originals/far-test-front.jpg"
print(f"Using input image: {input_path}")

# Copy into CodeFormer's expected input directory structure
import os
import shutil

os.makedirs('CodeFormer/inputs/my_test', exist_ok=True)
shutil.copy(input_path, f'CodeFormer/inputs/my_test/{os.path.basename(input_path)}')

## 6 - Run CodeFormer

`-w 0.7` balances fidelity to the original face vs. restoration quality (lower = closer to input, higher = more aggressive restoration). `--face_upsample` also upsamples the background around the face.

In [ ]:
%cd CodeFormer
!python inference_codeformer.py -w 0.7 --input_path inputs/my_test --output_path ../codeformer_results --face_upsample
%cd ..

import glob
codeformer_files = glob.glob('codeformer_results/final_results/*')
print("Output files:", codeformer_files)

## 7 - Before / after comparison

In [ ]:
import cv2
import matplotlib.pyplot as plt

original = cv2.imread(input_path)
codeformer_output = cv2.imread(codeformer_files[0])

def resize_to_height(img, height=300):
    ratio = height / img.shape[0]
    return cv2.resize(img, (int(img.shape[1] * ratio), height))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(resize_to_height(original), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original {original.shape[1]}x{original.shape[0]}")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(resize_to_height(codeformer_output), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"CodeFormer {codeformer_output.shape[1]}x{codeformer_output.shape[0]}")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 8 - Download result

In [ ]:
from google.colab import files
files.download(codeformer_files[0])